# Vector Component Plots (Zijderveld Diagrams)

## A Tutorial on Interpreting Progressive Demagnetization Data

When a rock is collected for paleomagnetic analysis, its natural remanent magnetization (NRM) is typically a **composite** of multiple magnetization components acquired at different times and under different conditions. The central challenge of paleomagnetic lab work is to separate these components and isolate the one of geological interest — usually the primary magnetization acquired when the rock formed.

**Progressive demagnetization** — either by alternating fields (AF) or by stepwise heating — systematically removes magnetization components in order of their stability (coercivity for AF, or blocking temperature for thermal). The resulting series of direction and intensity measurements at each demagnetization step can be visualized in several ways. The most powerful of these is the **vector component plot**, introduced by Zijderveld (1967), which simultaneously displays changes in both direction and intensity.

In this notebook, we will:

1. Understand how a magnetization vector is decomposed into Cartesian components
2. Build synthetic demagnetization data for a specimen with **two magnetization components**
3. Plot the data on **Zijderveld diagrams** using PmagPy
4. Plot the same data on **equal area stereonets** to see the directional changes
5. Explore how the two representations complement each other

In [ ]:
!pip install pmagpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm
from pmagpy import pmag, ipmag
from plot_zij import plot_zij

## From Directions to Cartesian Coordinates

A paleomagnetic measurement gives us three quantities: **declination** ($D$), **inclination** ($I$), and **intensity** ($M$). Together these describe a vector in 3D space. To work with vectors — and especially to add or subtract magnetization components — we convert them to **Cartesian coordinates**:

$$X = M \cos(I) \cos(D)$$

$$Y = M \cos(I) \sin(D)$$

$$Z = M \sin(I)$$

In the paleomagnetic convention:
- **X** points North
- **Y** points East
- **Z** points Down (positive inclination = downward)

PmagPy provides `pmag.dir2cart()` and `pmag.cart2dir()` to convert between these representations. Let's see how this works:

In [ ]:
# Convert a direction (dec, inc, intensity) to Cartesian coordinates
example_dir = [30.0, 45.0, 1.0]  # Dec=30, Inc=45, M=1.0
cart = pmag.dir2cart(example_dir)
print(f"Direction: Dec={example_dir[0]}°, Inc={example_dir[1]}°, M={example_dir[2]}")
print(f"Cartesian: X(N)={cart[0]:.4f}, Y(E)={cart[1]:.4f}, Z(Down)={cart[2]:.4f}")

# And back to directional form
back = pmag.cart2dir(cart)
print(f"\nRound-trip: Dec={back[0]:.1f}°, Inc={back[1]:.1f}°, M={back[2]:.4f}")

## Two-Component Magnetization: The Concept

Most rocks carry more than one magnetization component. A very common situation involves:

1. **A low-stability overprint** (Component A): This is often a viscous remanent magnetization (VRM) acquired gradually as the rock sits in the present-day geomagnetic field over millions of years. It has low coercivity and/or low blocking temperature, so it is removed first during demagnetization.

2. **A high-stability primary component** (Component B): This is the original magnetization acquired when the rock formed — for example, a TRM in a lava flow or a chemical remanent magnetization (CRM) in a sediment. It has higher coercivity and/or higher blocking temperature, so it survives to later demagnetization steps.

The **total NRM** is the vector sum of all components:

$$\vec{M}_{\text{NRM}} = \vec{M}_A + \vec{M}_B$$

During progressive demagnetization:
- At early steps, **Component A is removed** while B remains intact. The magnetization vector moves along a line in 3D, with the direction of that line corresponding to the direction of A.
- At later steps, **Component B is removed** and the vector moves along a different line that trends toward the origin. The direction of this second line is the direction of B — the component we want.

Let's build a synthetic example to see this clearly.

## Building Synthetic Demagnetization Data

We'll construct a specimen with two components:

- **Component A (overprint)**: Dec = 0°, Inc = 60°, intensity = 0.3 (pointing N and steeply down — like a present-day field overprint at a mid-northern-latitude site)
- **Component B (primary)**: Dec = 290°, Inc = 20°, intensity = 0.7 (pointing WNW and shallowly down — an ancient magnetization acquired when the continent occupied a different position relative to the geomagnetic pole)

The NRM is the vector sum of A and B. In the examples that follow, we model the coercivity spectrum of each component as a **log-Gaussian (lognormal) distribution** — the standard model for coercivity spectra in rock magnetism — and explore how **overlapping spectra** affect the appearance of Zijderveld diagrams and equal area projections.

In [ ]:
# Define the two components
comp_A_dir = [0.0, 60.0]    # overprint: North, steeply down
comp_B_dir = [290.0, 20.0]  # ancient: WNW, shallowly down
comp_A_intensity = 0.30
comp_B_intensity = 0.70

# Convert to Cartesian
comp_A_cart = np.array(pmag.dir2cart([comp_A_dir[0], comp_A_dir[1], comp_A_intensity]))
comp_B_cart = np.array(pmag.dir2cart([comp_B_dir[0], comp_B_dir[1], comp_B_intensity]))

print(f"Component A (overprint): Dec={comp_A_dir[0]}°, Inc={comp_A_dir[1]}°, M={comp_A_intensity}")
print(f"  Cartesian: X={comp_A_cart[0]:.4f}, Y={comp_A_cart[1]:.4f}, Z={comp_A_cart[2]:.4f}")
print(f"\nComponent B (primary):   Dec={comp_B_dir[0]}°, Inc={comp_B_dir[1]}°, M={comp_B_intensity}")
print(f"  Cartesian: X={comp_B_cart[0]:.4f}, Y={comp_B_cart[1]:.4f}, Z={comp_B_cart[2]:.4f}")

# The NRM is the vector sum
nrm_cart = comp_A_cart + comp_B_cart
nrm_dir = pmag.cart2dir(nrm_cart)
nrm_dec = nrm_dir[0]  # used as the Zijderveld rotation angle
print(f"\nNRM (A+B):    Dec={nrm_dir[0]:.1f}°, Inc={nrm_dir[1]:.1f}°, M={nrm_dir[2]:.4f}")
print("\nNotice that the NRM direction differs from both A and B — it is their vector sum.")
print(f"We will project Zijderveld diagrams along the NRM declination ({nrm_dec:.1f}°)")

In [ ]:
# AF demagnetization steps (mT)
af_steps = np.array([0, 2, 5, 8, 10, 15, 20, 25, 30, 35, 40, 45, 50,
                     60, 70, 80, 90, 100, 120, 140, 160, 180, 200])
af_fine = np.linspace(0.1, 200, 500)  # for smooth spectrum curves (avoid x=0 for lognorm)


def generate_demag_data(steps, comp_A_cart, comp_B_cart,
                        median_A, dp_A, median_B, dp_B,
                        verbose=False):
    """Generate synthetic demagnetization data with log-Gaussian coercivity spectra.

    Each component's coercivity (or unblocking temperature) spectrum is modeled
    as a lognormal distribution parameterized by its median and dispersion
    parameter (sigma of the underlying normal distribution in log-space).

    Args:
        steps: Demagnetization levels (mT or °C).
        comp_A_cart, comp_B_cart: Cartesian vectors for components A and B.
        median_A, dp_A: Median and dispersion parameter of component A's spectrum.
        median_B, dp_B: Median and dispersion parameter of component B's spectrum.
        verbose: If True, print a table of step, frac_A, frac_B, Dec, Inc, M.

    Returns:
        demag_data: Datablock for plot_zij [step, dec, inc, M, type, quality].
        frac_A, frac_B: Fraction of each component remaining at each step.
    """
    frac_A = 1.0 - lognorm.cdf(steps, s=dp_A, scale=median_A)
    frac_B = 1.0 - lognorm.cdf(steps, s=dp_B, scale=median_B)

    if verbose:
        print(f"{'Step':>6s}  {'frac_A':>6s}  {'frac_B':>6s}  {'Dec':>7s}  {'Inc':>7s}  {'M':>8s}")
        print("-" * 50)

    demag_data = []
    for i, step in enumerate(steps):
        total_cart = frac_A[i] * comp_A_cart + frac_B[i] * comp_B_cart
        total_dir = pmag.cart2dir(total_cart)
        dec, inc, intensity = total_dir[0], total_dir[1], total_dir[2]
        if np.isnan(dec):
            dec = 0.0
        if np.isnan(inc):
            inc = 0.0
        demag_data.append([step, dec, inc, intensity, 'DA', 'g'])
        if verbose:
            print(f"{step:6.1f}  {frac_A[i]:6.3f}  {frac_B[i]:6.3f}  "
                  f"{dec:7.1f}  {inc:7.1f}  {intensity:8.4f}")

    if verbose:
        print(f"\n{len(demag_data)} demagnetization steps generated.")
    return demag_data, frac_A, frac_B


def plot_spectrum(ax, x_fine, median_A, dp_A, median_B, dp_B,
                  xlabel='AF level (mT)', ylabel='dM/dB'):
    """Plot log-Gaussian coercivity/unblocking spectra for components A and B."""
    spec_A = comp_A_intensity * lognorm.pdf(x_fine, s=dp_A, scale=median_A)
    spec_B = comp_B_intensity * lognorm.pdf(x_fine, s=dp_B, scale=median_B)
    ax.fill_between(x_fine, spec_A, alpha=0.4, color='#D55E00')
    ax.fill_between(x_fine, spec_B, alpha=0.4, color='#0072B2')
    ax.plot(x_fine, spec_A, color='#D55E00', linewidth=1.5, label='$J_A$')
    ax.plot(x_fine, spec_B, color='#0072B2', linewidth=1.5, label='$J_B$')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=11, loc='upper right')
    ax.set_xlim(x_fine[0], x_fine[-1])


def plot_intensity(ax, steps, demag_data, xlabel='AF level (mT)'):
    """Plot normalized intensity decay curve."""
    intensities = np.array([row[3] for row in demag_data])
    norm_int = intensities / intensities[0] if intensities[0] > 0 else intensities
    ax.plot(steps, norm_int, '-ok', markersize=4, linewidth=1.5)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('M / M$_{NRM}$', fontsize=11)
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, alpha=0.2)


def plot_equal_area(ax, demag_data):
    """Plot equal area projection for demagnetization data using PmagPy."""
    plt.sca(ax)
    ipmag.plot_net(ax=ax)
    decs = [row[1] for row in demag_data]
    incs = [row[2] for row in demag_data]
    ipmag.plot_di(dec=decs, inc=incs, color='gray', markersize=30)
    ipmag.plot_di(dec=[comp_A_dir[0]], inc=[comp_A_dir[1]],
                  color='#D55E00', marker='*', markersize=200)
    ipmag.plot_di(dec=[comp_B_dir[0]], inc=[comp_B_dir[1]],
                  color='#0072B2', marker='*', markersize=200)
    ipmag.plot_di(dec=[decs[0]], inc=[incs[0]],
                  color='black', marker='D', markersize=80)


print(f"AF steps: {af_steps}")
print("Helper functions defined: generate_demag_data(), plot_spectrum(),")
print("  plot_intensity(), plot_equal_area()")

## The Zijderveld Diagram and Equal Area Projection

The **Zijderveld diagram** (Zijderveld, 1967) projects the 3D magnetization vector onto two orthogonal planes plotted on shared axes:

- **Solid circles**: horizontal projection (declination + horizontal intensity)
- **Open squares**: vertical projection (inclination + vertical intensity)

The key property is that **removal of a single magnetization component traces a straight line**. Two non-overlapping components produce two distinct linear segments, with the high-stability segment trending toward the origin.

Following Tauxe (2010), we project along the **NRM declination** — because the NRM is the vector sum of all components, this guarantees that every component has a nonzero projection onto the x-axis.

The **equal area projection** (stereonet) shows the directional path during demagnetization. As the overprint is removed, the direction migrates along a great circle toward the primary component, then stabilizes.

Below we examine three cases with progressively increasing overlap between the coercivity spectra of Components A and B.

In [ ]:
# Case 1: Non-overlapping coercivity spectra
# Component A: median 12 mT, narrow (dp=0.4)
# Component B: median 100 mT, narrow (dp=0.3)
print("Case 1: Non-overlapping log-Gaussian spectra\n")
data1, frac_A1, frac_B1 = generate_demag_data(
    af_steps, comp_A_cart, comp_B_cart,
    median_A=12, dp_A=0.4, median_B=100, dp_B=0.3,
    verbose=True)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 11))

plot_spectrum(ax1, af_fine, 12, 0.4, 100, 0.3)
ax1.set_title('Coercivity Spectra', fontsize=12)

plot_intensity(ax2, af_steps, data1)
ax2.set_title('Intensity Decay', fontsize=12)

plot_zij(None, data1, angle=nrm_dec, s='Non-overlapping',
         label_list=[0, 10, 30, 50, 100, 200], unit='mT', ax=ax3)

plot_equal_area(ax4, data1)
ax4.set_title('Equal Area', fontsize=12)

plt.tight_layout()
plt.show()

### Case 1: Non-overlapping spectra

When the coercivity spectra do not overlap, the Zijderveld diagram shows two **distinct linear segments**: the first corresponds to removal of the overprint, and the second trends **toward the origin**, revealing the primary magnetization. The transition is sharp.

On the equal area plot, the direction migrates smoothly from the NRM (black diamond) toward Component B (blue star) as A is removed, then stabilizes. This is the ideal case — both components are cleanly resolved and PCA line fits yield accurate directions.

### Case 2: Partially Overlapping Coercivity Spectra

In many real specimens, the coercivity spectra of the two components overlap partially. Over some range of demagnetization levels, **both components are being removed simultaneously**. The vector path in this overlap zone is no longer a straight line — it follows a **curved trajectory** as the relative contributions of A and B change together.

In [ ]:
# Case 2: Partially overlapping coercivity spectra
# Component A: median 20 mT, broader (dp=0.5)
# Component B: median 70 mT, moderate (dp=0.4)
print("Case 2: Partially overlapping log-Gaussian spectra\n")
data2, frac_A2, frac_B2 = generate_demag_data(
    af_steps, comp_A_cart, comp_B_cart,
    median_A=20, dp_A=0.5, median_B=70, dp_B=0.4,
    verbose=True)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 11))

plot_spectrum(ax1, af_fine, 20, 0.5, 70, 0.4)
ax1.set_title('Coercivity Spectra', fontsize=12)

plot_intensity(ax2, af_steps, data2)
ax2.set_title('Intensity Decay', fontsize=12)

plot_zij(None, data2, angle=nrm_dec, s='Partial overlap',
         label_list=[0, 10, 30, 50, 100, 200], unit='mT', ax=ax3)

plot_equal_area(ax4, data2)
ax4.set_title('Equal Area', fontsize=12)

plt.tight_layout()
plt.show()

With partial overlap, the Zijderveld diagram shows two approximately linear segments connected by a **curved transition zone**. The linear segments at low and high demagnetization levels still reveal the directions of Components A and B, but the intermediate points deviate from both lines.

On the equal area plot, the directional path still migrates from the NRM toward Component B, but the path is less direct through the overlap zone. A PCA fit to the high-stability segment can still recover a good estimate of Component B's direction, provided the analyst selects steps clearly beyond the overlap zone.

### Case 3: Strongly Overlapping Coercivity Spectra

When the spectra overlap extensively, there may be **no demagnetization level at which only one component is being removed**. Both components unblock together over most of the treatment range, producing a continuously curved vector path on the Zijderveld diagram.

In [ ]:
# Case 3: Strongly overlapping coercivity spectra
# Component A: median 30 mT, broad (dp=0.6)
# Component B: median 50 mT, broad (dp=0.5)
print("Case 3: Strongly overlapping log-Gaussian spectra\n")
data3, frac_A3, frac_B3 = generate_demag_data(
    af_steps, comp_A_cart, comp_B_cart,
    median_A=30, dp_A=0.6, median_B=50, dp_B=0.5,
    verbose=True)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 11))

plot_spectrum(ax1, af_fine, 30, 0.6, 50, 0.5)
ax1.set_title('Coercivity Spectra', fontsize=12)

plot_intensity(ax2, af_steps, data3)
ax2.set_title('Intensity Decay', fontsize=12)

plot_zij(None, data3, angle=nrm_dec, s='Strong overlap',
         label_list=[0, 10, 30, 50, 100, 200], unit='mT', ax=ax3)

plot_equal_area(ax4, data3)
ax4.set_title('Equal Area', fontsize=12)

plt.tight_layout()
plt.show()

With strong overlap, the Zijderveld diagram shows a **continuously curved path** with no clearly linear segments. Neither component is isolated at any demagnetization step, making it difficult to determine either direction reliably from PCA line fits.

On the equal area plot, the directional path trends generally from the NRM toward Component B, but without a clear stabilization point. The direction at the highest steps approaches Component B's true direction but remains contaminated by residual contributions from Component A.

This is the most challenging scenario for paleomagnetic interpretation. Techniques such as **great circle analysis** can sometimes extract useful directional information from curved demagnetization paths, even when PCA line fits are unreliable.

## Implications for Paleomagnetic Analysis

The three cases above illustrate a fundamental challenge in paleomagnetism: **the quality of component separation depends on the degree of overlap between coercivity (or unblocking temperature) spectra**.

- **Non-overlapping spectra** produce clean, linear segments on the Zijderveld diagram. PCA line fits yield accurate directions with low MAD values.
- **Partially overlapping spectra** produce curved transition zones, but the endpoints of the demagnetization path may still define usable linear segments. Careful selection of demagnetization steps for PCA analysis can recover good directional estimates.
- **Strongly overlapping spectra** produce continuously curved paths. PCA line fits are unreliable, and alternative techniques (great circle analysis, remagnetization circle methods) may be needed.

The degree of overlap depends on the mineralogy, grain size distribution, and thermal history of the rock. Thermal demagnetization and AF demagnetization may resolve components differently — a specimen with overlapping AF spectra may have well-separated unblocking temperature spectra, and vice versa. This is why it is common practice to demagnetize companion specimens using both methods.

## Interactive Explorer

Use the sliders below to adjust the **median coercivity** and **dispersion parameter (DP)** of each component's log-Gaussian spectrum. The coercivity spectrum, Zijderveld diagram, and equal area projection update automatically. Try reproducing the three cases above, then explore what happens when you push the spectra closer together or make them broader.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Sliders for Component A (overprint)
median_A_slider = widgets.FloatSlider(
    value=20, min=2, max=100, step=1,
    description='Median A (mT):',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'))
dp_A_slider = widgets.FloatSlider(
    value=0.4, min=0.1, max=1.0, step=0.05,
    description='DP A:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'))

# Sliders for Component B (primary)
median_B_slider = widgets.FloatSlider(
    value=80, min=10, max=200, step=1,
    description='Median B (mT):',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'))
dp_B_slider = widgets.FloatSlider(
    value=0.4, min=0.1, max=1.0, step=0.05,
    description='DP B:',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'))


def _plot_equal_area_widget(ax, demag_data):
    """Equal area plot using pmag.dimap + ax methods.

    Avoids ipmag.plot_di's plt.tight_layout() calls which trigger
    intermediate renders inside ipywidgets Output widgets.
    """
    # Draw net circle
    Dcirc = np.arange(0, 361.)
    Xcirc = [pmag.dimap(d, 0)[0] for d in Dcirc]
    Ycirc = [pmag.dimap(d, 0)[1] for d in Dcirc]
    ax.plot(Xcirc, Ycirc, 'k')
    # Tick marks
    for D in range(0, 360, 10):
        Xtick = [pmag.dimap(D, i)[0] for i in range(4)]
        Ytick = [pmag.dimap(D, i)[1] for i in range(4)]
        ax.plot(Xtick, Ytick, 'k')

    def _plot_point(ax, dec, inc, **kwargs):
        xy = pmag.dimap(dec, inc)
        fc = kwargs.pop('facecolor', kwargs.get('color', 'k'))
        if inc < 0:
            fc = 'none'
        ax.scatter(xy[0], xy[1], facecolors=fc, edgecolors=kwargs.pop('edgecolor', fc),
                   s=kwargs.pop('s', 30), marker=kwargs.pop('marker', 'o'),
                   zorder=kwargs.pop('zorder', 2))

    # Demag path
    decs = [row[1] for row in demag_data]
    incs = [row[2] for row in demag_data]
    for d, i in zip(decs, incs):
        _plot_point(ax, d, i, facecolor='gray', edgecolor='gray', s=30)

    # Component direction stars and NRM diamond
    _plot_point(ax, comp_A_dir[0], comp_A_dir[1],
                facecolor='#D55E00', edgecolor='#D55E00', marker='*', s=200, zorder=3)
    _plot_point(ax, comp_B_dir[0], comp_B_dir[1],
                facecolor='#0072B2', edgecolor='#0072B2', marker='*', s=200, zorder=3)
    _plot_point(ax, decs[0], incs[0],
                facecolor='black', edgecolor='black', marker='D', s=80, zorder=3)

    ax.set_aspect('equal')
    ax.axis((-1.05, 1.05, -1.05, 1.05))
    ax.axis('off')


def update_plots(median_A, dp_A, median_B, dp_B):
    data, frac_A, frac_B = generate_demag_data(
        af_steps, comp_A_cart, comp_B_cart,
        median_A, dp_A, median_B, dp_B)

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 11))

    plot_spectrum(ax1, af_fine, median_A, dp_A, median_B, dp_B)
    ax1.set_title('Coercivity Spectra', fontsize=12)

    plot_intensity(ax2, af_steps, data)
    ax2.set_title('Intensity Decay', fontsize=12)

    plot_zij(None, data, angle=nrm_dec, s='Interactive',
             label_list=[0, 10, 30, 50, 100, 200], unit='mT', ax=ax3)

    _plot_equal_area_widget(ax4, data)
    ax4.set_title('Equal Area', fontsize=12)

    fig.tight_layout()
    display(fig)
    plt.close(fig)


out = widgets.interactive_output(update_plots, {
    'median_A': median_A_slider,
    'dp_A': dp_A_slider,
    'median_B': median_B_slider,
    'dp_B': dp_B_slider,
})

controls = widgets.VBox([
    widgets.HTML('<b style="color:#D55E00">Component A (overprint)</b>'),
    widgets.HBox([median_A_slider, dp_A_slider]),
    widgets.HTML('<b style="color:#0072B2">Component B (primary)</b>'),
    widgets.HBox([median_B_slider, dp_B_slider]),
])

display(controls, out)

## Thermal Demagnetization Example

The same principles apply to **thermal demagnetization**, where the controlling parameter is **blocking temperature** rather than coercivity. Here we show a non-overlapping case where the overprint is removed by ~300°C and the primary component persists up to ~580°C (the Curie temperature of magnetite).

In [ ]:
# Thermal demagnetization with non-overlapping unblocking temperature spectra
temp_steps = np.array([20, 100, 150, 200, 250, 300,
                       350, 400, 450, 500, 530, 550, 565, 575, 580])
temp_fine = np.linspace(1, 600, 500)

print("Thermal demagnetization: Non-overlapping unblocking T spectra\n")
thermal_data, frac_A_th, frac_B_th = generate_demag_data(
    temp_steps, comp_A_cart, comp_B_cart,
    median_A=150, dp_A=0.3, median_B=500, dp_B=0.1,
    verbose=True)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 11))

plot_spectrum(ax1, temp_fine, 150, 0.3, 500, 0.1,
              xlabel='Temperature (°C)', ylabel='dM/dT')
ax1.set_title('Unblocking T Spectra', fontsize=12)

plot_intensity(ax2, temp_steps, thermal_data, xlabel='Temperature (°C)')
ax2.set_title('Intensity Decay', fontsize=12)

plot_zij(None, thermal_data, angle=nrm_dec, s='Thermal demag',
         label_list=[20, 150, 300, 500, 580], unit='°C', ax=ax3)

plot_equal_area(ax4, thermal_data)
ax4.set_title('Equal Area', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Summary figure — also available as standalone script: overlapping_spectra_figure.py
# Equal-intensity components (50/50) for illustrative clarity
from plot_zij import overlay_components

comp_A_dir_eq = [0.0, 60.0]     # overprint: North, steeply down
comp_B_dir_eq = [270.0, 20.0]   # ancient: W, shallowly down
int_eq = 0.50
A_cart_eq = np.array(pmag.dir2cart([comp_A_dir_eq[0], comp_A_dir_eq[1], int_eq]))
B_cart_eq = np.array(pmag.dir2cart([comp_B_dir_eq[0], comp_B_dir_eq[1], int_eq]))
nrm_dec_eq = pmag.cart2dir(A_cart_eq + B_cart_eq)[0]

cases = [
    {'median_A': 12, 'dp_A': 0.4, 'median_B': 100, 'dp_B': 0.3,
     'row_title': 'Non-overlapping spectra'},
    {'median_A': 20, 'dp_A': 0.5, 'median_B': 70,  'dp_B': 0.4,
     'row_title': 'Small overlap'},
    {'median_A': 30, 'dp_A': 0.6, 'median_B': 50,  'dp_B': 0.5,
     'row_title': 'Large overlap'},
]

fig, axes = plt.subplots(3, 2, figsize=(10, 9))

for row, case in enumerate(cases):
    ax_spec = axes[row, 0]
    ax_zij = axes[row, 1]

    data_eq = generate_demag_data(
        af_steps, A_cart_eq, B_cart_eq,
        case['median_A'], case['dp_A'], case['median_B'], case['dp_B'])

    # Left: coercivity spectra
    spec_A = int_eq * lognorm.pdf(af_fine, s=case['dp_A'], scale=case['median_A'])
    spec_B = int_eq * lognorm.pdf(af_fine, s=case['dp_B'], scale=case['median_B'])
    ax_spec.fill_between(af_fine, spec_A, alpha=0.3, color='#D55E00')
    ax_spec.fill_between(af_fine, spec_B, alpha=0.3, color='#0072B2')
    ax_spec.plot(af_fine, spec_A, color='#D55E00', linewidth=1.5)
    ax_spec.plot(af_fine, spec_B, color='#0072B2', linewidth=1.5)
    for median, dp, label, color in [
        (case['median_A'], case['dp_A'], '$M_A$', '#D55E00'),
        (case['median_B'], case['dp_B'], '$M_B$', '#0072B2'),
    ]:
        peak_y = int_eq * lognorm.pdf(median, s=dp, scale=median)
        ax_spec.text(median, peak_y * 0.45, label, ha='center', va='center',
                     fontsize=13, fontweight='bold', color=color)
    ax_spec.set_ylabel('dM/dB', fontsize=11)
    ax_spec.set_xlim(0.1, 200)
    ax_spec.set_ylim(bottom=0)
    ax_spec.set_title(case['row_title'], fontsize=12, loc='left')
    if row == 2:
        ax_spec.set_xlabel('AF level (mT)', fontsize=11)
    else:
        ax_spec.set_xticklabels([])

    # Right: Zijderveld diagram
    plot_zij(None, data_eq, angle=nrm_dec_eq, s='',
             label_list=[0, 15, 30, 60], unit='mT',
             ax=ax_zij, title='')

    # Show vector addition on all cases
    nrm_intensity_eq = data_eq[0][3]
    overlay_components(ax_zij, A_cart_eq, B_cart_eq,
                       angle=nrm_dec_eq, norm_factor=1.0 / nrm_intensity_eq)

# Force all Zij panels to share the same axis limits
zij_lims = axes[0, 1].axis()
for row in range(1, 3):
    axes[row, 1].axis(zij_lims)

fig.tight_layout()
plt.show()

## Summary

### Key concepts

- The **natural remanent magnetization (NRM)** of a rock is typically the vector sum of multiple components acquired at different times.

- **Progressive demagnetization** (AF or thermal) removes components in order of increasing stability, allowing them to be separated.

- The **Zijderveld diagram** plots the endpoint of the magnetization vector projected onto two orthogonal planes (horizontal and vertical). Projecting along the NRM declination (Tauxe, 2010) ensures all components have nonzero x-projections.

- A **single magnetization component** appears as a **straight line** on the Zijderveld plot. Two non-overlapping components produce two distinct linear segments.

- When coercivity or unblocking temperature spectra **overlap**, the Zijderveld path becomes **curved** in the overlap zone, making component separation more difficult.

- **Equal area plots** show the directional path during demagnetization. The direction migrates as the overprint is removed, then stabilizes once only the ChRM remains — provided the spectra are well separated.

- Both plot types are complementary: the Zijderveld diagram shows direction and intensity together, while the equal area plot provides a clearer view of directional changes.

### References

- Kirschvink, J.L. (1980). The least-squares line and plane and the analysis of palaeomagnetic data. *Geophysical Journal International*, 62(3), 699–718.
- Tauxe, L. (2010). *Essentials of Paleomagnetism*. University of California Press.
- Tauxe, L., Shaar, R., Jonestrask, L., et al. (2016). PmagPy: Software package for paleomagnetic data analysis and a bridge to the Magnetics Information Consortium (MagIC) Database. *Geochemistry, Geophysics, Geosystems*, 17(6), 2450–2463.
- Zijderveld, J.D.A. (1967). AC demagnetization of rocks: analysis of results. In: *Methods in Paleomagnetism*, Elsevier, pp. 254–286.